# Building an Automated Data Pipeline (ETL + Airflow / Luigi Orchestration)
**Dataset:** Breast Cancer Wisconsin Diagnostic Clinical Feed ($584 \rightarrow 569$ Patients)

### Pipeline Architecture & Stages:
1. **Stage 1 — Extract (`extract_data`):** Ingest raw upstream clinical feed (`584` rows $\times$ `35` columns).
2. **Stage 2 — Clean & Validate (`clean_and_validate`):** Remove `15` duplicate patient records, impute `27` missing values via median imputation, cap `612` extreme cell-measurement outliers using the $1.5 \times \text{IQR}$ rule, and enforce automated Data Quality Gate assertions.
3. **Stage 3 — Transform & Engineer (`transform_and_engineer`):** Construct `5` domain geometric biomarkers, apply Ordinal & One-Hot Encoding, and standardize `35` continuous predictors ($79$ total columns).
4. **Stage 4 — Load (`load_to_warehouse`):** Persist model-ready tables (`fact_breast_cancer_diagnostics` and `agg_diagnosis_by_size_grade`) into a relational **SQLite Data Warehouse** (`clinical_data_warehouse.db`) and export an execution audit log.

In [ ]:
import os
import sys
import json
import sqlite3
import pandas as pd

if os.path.exists('../src/etl_pipeline.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from src.etl_pipeline import run_pipeline
audit_log = run_pipeline()
print('\n=== PIPELINE AUDIT LOG ===')
print(json.dumps(audit_log, indent=2))

## Query the Loaded SQLite Data Warehouse

In [ ]:
conn = sqlite3.connect('data/processed/clinical_data_warehouse.db')
df_agg = pd.read_sql_query('SELECT * FROM agg_diagnosis_by_size_grade', conn)
display(df_agg)

df_sample = pd.read_sql_query(
    'SELECT patient_id, diagnosis_label, tumor_size_grade, mean_radius, area_perimeter_ratio, nuclear_malformation_score '
    'FROM fact_breast_cancer_diagnostics LIMIT 8', conn
)
display(df_sample)
conn.close()